In [1]:
from pyspark import SparkContext

In [2]:
sc = SparkContext(master='local', appName='transformacionesYAcciones')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 13:30:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
rdd = sc.parallelize([1,2,3])
type(rdd)

pyspark.core.rdd.RDD

In [4]:
rdd.collect()

[1, 2, 3]

In [5]:
sc

<SparkContext master=local appName=transformacionesYAcciones>

In [6]:
!ls /home/jovyan/data/

deporte.csv	 deportistaError.csv  modelo_relacional.jpg
deportista2.csv  evento.csv	      paises.csv
deportista.csv	 juegos.csv	      resultados.csv


In [7]:
path = "/home/jovyan/data/"

In [8]:
equiposOlimpicosRDD = sc.textFile(path+"paises.csv") \
.map(lambda line : line.split(","))

In [9]:
equiposOlimpicosRDD.take(15)

[['id', 'equipo', 'sigla'],
 ['1', '30. Februar', 'AUT'],
 ['2', 'A North American Team', 'MEX'],
 ['3', 'Acipactli', 'MEX'],
 ['4', 'Acturus', 'ARG'],
 ['5', 'Afghanistan', 'AFG'],
 ['6', 'Akatonbo', 'IRL'],
 ['7', 'Alain IV', 'SUI'],
 ['8', 'Albania', 'ALB'],
 ['9', 'Alcaid', 'POR'],
 ['10', 'Alcyon-6', 'FRA'],
 ['11', 'Alcyon-7', 'FRA'],
 ['12', 'Aldebaran', 'ITA'],
 ['13', 'Aldebaran II', 'ITA'],
 ['14', 'Aletta', 'IRL']]

In [10]:
equiposOlimpicosRDD.map(lambda x: (x[2])).distinct().count()

231

In [11]:
equiposOlimpicosRDD.map(lambda x: (x[2], x[1])).groupByKey().mapValues(len).take(5)

[('sigla', 1), ('AUT', 11), ('MEX', 9), ('ARG', 18), ('AFG', 1)]

In [12]:
equiposOlimpicosRDD.map(lambda x: (x[2], x[1])).groupByKey() \
    .mapValues(list).take(5)

[('sigla', ['equipo']),
 ('AUT',
  ['30. Februar',
   'Austria',
   'Austria-1',
   'Austria-2',
   'Breslau',
   'Brigantia',
   'Donar III',
   'Evita VI',
   'May-Be 1960',
   '"R.-V. Germania; Leitmeritz"',
   'Surprise']),
 ('MEX',
  ['A North American Team',
   'Acipactli',
   'Chamukina',
   'Mexico',
   'Mexico-1',
   'Mexico-2',
   'Nausikaa 4',
   'Tlaloc',
   'Xolotl']),
 ('ARG',
  ['Acturus',
   'Antares',
   'Arcturus',
   'Ardilla',
   'Argentina',
   'Argentina-1',
   'Argentina-2',
   'Blue Red',
   'Covunco III',
   'Cupidon III',
   'Djinn',
   'Gullvinge',
   'Matrero II',
   'Mizar',
   'Pampero',
   'Rampage',
   'Tango',
   'Wiking']),
 ('AFG', ['Afghanistan'])]

In [13]:
equiposArgentinos = equiposOlimpicosRDD.filter(lambda l : "ARG" in l)
equiposArgentinos.collect()

[['4', 'Acturus', 'ARG'],
 ['37', 'Antares', 'ARG'],
 ['42', 'Arcturus', 'ARG'],
 ['43', 'Ardilla', 'ARG'],
 ['45', 'Argentina', 'ARG'],
 ['46', 'Argentina-1', 'ARG'],
 ['47', 'Argentina-2', 'ARG'],
 ['119', 'Blue Red', 'ARG'],
 ['238', 'Covunco III', 'ARG'],
 ['252', 'Cupidon III', 'ARG'],
 ['288', 'Djinn', 'ARG'],
 ['436', 'Gullvinge', 'ARG'],
 ['644', 'Matrero II', 'ARG'],
 ['672', 'Mizar', 'ARG'],
 ['774', 'Pampero', 'ARG'],
 ['843', 'Rampage', 'ARG'],
 ['1031', 'Tango', 'ARG'],
 ['1162', 'Wiking', 'ARG']]

In [14]:
equiposOlimpicosRDD.countApprox(20)

1185

In [15]:
deportistaOlimpicoRDD = sc.textFile(path+"deportista.csv") \
    .map(lambda l : l.split(","))
deportistaOlimpicoRDD2 = sc.textFile(path+"deportista2.csv") \
    .map(lambda l : l.split(","))

In [16]:
deportistaOlimpicoRDD = deportistaOlimpicoRDD \
    .union(deportistaOlimpicoRDD2)

In [17]:
deportistaOlimpicoRDD.count()

135572

In [18]:
deportistaOlimpicoRDD.top(2)

[['deportista_id', 'nombre', 'genero', 'edad', 'altura', 'peso', 'equipo_id'],
 ['99999', 'Alexander Grant Alick Rennie', '1', '32', '182', '71', '967']]

In [19]:
deportistaOlimpicoRDD.map(lambda l: [l[6], l[:6]]) \
.join(equiposOlimpicosRDD.map(lambda x: [x[0], x[2]])) \
.takeSample(False,5,25)

[('970', (['68062', 'Lee MinHui', '2', '28', '174', '65'], 'KOR')),
 ('154', (['39161', 'Angel Merdzhanov Gavrilov', '1', '24', '0', '0'], 'BUL')),
 ('1084',
  (['62843', 'Olha Vasylivna Korobka', '2', '18', '181', '167'], 'UKR')),
 ('678', (['97550', 'Puntsagiin Skhbat', '1', '24', '174', '82'], 'MGL')),
 ('1096', (['106789', 'Hugo Scherzer', '1', '43', '0', '0'], 'USA'))]

In [20]:
resultado = sc.textFile(path+"resultados.csv") \
.map(lambda l : l.split(","))

In [21]:
resultadoGanador = resultado.filter(lambda l : 'NA' not in l[1])

In [22]:
resultadoGanador.take(2)

[['resultado_id', 'medalla', 'deportista_id', 'juego_id', 'evento_id'],
 ['4', 'Gold', '4', '2', '4']]

In [23]:
deportistaPais = deportistaOlimpicoRDD \
.map(lambda l :[l[-1], l[:-1]]) \
.join(equiposOlimpicosRDD.map(lambda x :[x[0], x[2]]))

In [32]:
deportistaPais.take(6)

[('199', (['1', 'A Dijiang', '1', '24', '180', '80'], 'CHN')),
 ('199', (['2', 'A Lamusi', '1', '23', '170', '60'], 'CHN')),
 ('199', (['602', 'Abudoureheman', '1', '22', '182', '75'], 'CHN')),
 ('199', (['1463', 'Ai Linuer', '1', '25', '160', '62'], 'CHN')),
 ('199', (['1464', 'Ai Yanhan', '2', '14', '168', '54'], 'CHN')),
 ('199', (['3605', 'An Weijiang', '1', '22', '178', '72'], 'CHN'))]

In [33]:
deportistaPais.map(lambda x: (x[1][0][0], x)).join(resultadoGanador.map(lambda y: (y[2], y[1]))).take(5)

[('7597',
  (('199', (['7597', 'Bao Yingying', '2', '24', '172', '67'], 'CHN')),
   'Silver')),
 ('17282',
  (('199', (['17282', 'Cai Huijue', '2', '16', '174', '63'], 'CHN')),
   'Bronze')),
 ('17996',
  (('199', (['17996', 'Cao Mianying', '2', '21', '176', '71'], 'CHN')),
   'Silver')),
 ('19779',
  (('199', (['19779', 'Chang Si', '2', '25', '170', '56'], 'CHN')), 'Silver')),
 ('19791',
  (('199', (['19791', 'Chang Yongxiang', '1', '24', '178', '74'], 'CHN')),
   'Silver'))]

In [34]:
valoresMedallas = {'Gold' : 7, 'Silver' : 5, 'Bronze' : 4}

In [35]:
paisesMedallas = deportistaPais.join(resultadoGanador)

In [36]:
paisesMedallas.map(lambda x: (x[1][0][-1], valoresMedallas[x[1][1]]))

PythonRDD[96] at RDD at PythonRDD.scala:56

In [37]:
from operator import add
rdd_total = paisesMedallas.map(lambda x: (x[1][0][-1], valoresMedallas[x[1][1]]))
respuesta = rdd_total.reduceByKey(add) \
.sortBy(lambda x :x[1], ascending=False)

In [38]:
respuesta.take(5)

[('CAN', 32538), ('ARG', 12520), ('HUN', 10860), ('MEX', 6124), ('RSA', 3788)]

In [39]:
def eliminaEncabezados(indice, iterador):
    return iter(list(iterador)[1:])

In [41]:
deportistaOlimpicoRDD = deportistaOlimpicoRDD.mapPartitionsWithIndex(eliminaEncabezados)

In [42]:
deportistaOlimpicoRDD.take(5)

[['1', 'A Dijiang', '1', '24', '180', '80', '199'],
 ['2', 'A Lamusi', '1', '23', '170', '60', '199'],
 ['3', 'Gunnar Nielsen Aaby', '1', '24', '0', '0', '273'],
 ['4', 'Edgar Lindenau Aabye', '1', '34', '0', '0', '278'],
 ['5', 'Christine Jacoba Aaftink', '2', '21', '185', '82', '705']]

In [43]:
deportistaOlimpicoRDD = deportistaOlimpicoRDD.map(lambda l : (
    int(l[0]),
    l[1],
    int(l[2]),
    int(l[3]),
    int(l[4]),
    float(l[5]),
    int(l[6])
))

In [49]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
from pyspark.sql.types import Row
from pyspark.sql import SQLContext

In [53]:
sqlContext = SQLContext(sc)

In [54]:
schema = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("genero", IntegerType(), False),
    StructField("edad", IntegerType(), False),
    StructField("altura", IntegerType(), False),
    StructField("peso", FloatType(), False),
    StructField("equipo_id", IntegerType(), False),
])

In [55]:
deportistaDF = sqlContext.createDataFrame(deportistaOlimpicoRDD, schema).show(5)

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|80.0|      199|
|            2|            A Lamusi|     1|  23|   170|60.0|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0| 0.0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0| 0.0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|82.0|      705|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows


In [56]:
sc.stop()